# LitServe + Cloudflare Tunnel (Qwen 1.5B Coder + LoRA) x1

In [19]:
MODEL_ZIP_URL = 'https://edtechblob.blob.core.windows.net/models/models_v2.zip?se=2035-12-25T15%3A59%3A22Z&sp=r&sv=2025-11-05&sr=b&sig=G%2BFzkenKR2IWsVySDM7E03/1lzwfbbBAuXDXTBge0IU%3D'

MODEL_CONFIG = {
    'base_model': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    'lora_folder': 'qwen-1.5b',
    'default_system_prompt': 'Bạn là trợ lý AI giảng dạy lập trình. Giải thích đúng trọng tâm câu hỏi. Trả lời tiếng Việt, code có comment tiếng Việt.'
}

DB_CONFIG = {
    'host': '20.255.153.62',
    'port': 5432,
    'user': 'postgres',
    'password': 'NguyenDuc@163',
    'database': 'edtechdb'
}

AZURE_CONFIG = {
    'account_name': 'edtechblob',
    'account_key': 'W8UpS5mF7imTY2+AQXEuY8UsJNbWjXDHPWHYRzye8oVJNV9iMU98/6/Lf/XL3SkHhMttQYL+VCO/+AStt/G4XQ==',
    'container_name': 'test',
    'thumbnail_container': 'thumbnails'
}

CLOUDFLARED_TOKEN = "eyJhIjoiMjU5ODZjNDVkZDU3NzUzYTMwMDk5NjRkMDQwZTE3ZmYiLCJ0IjoiYzhhZWNiY2UtYzExYS00OGMwLWFhYTgtYzM4YjdhNmNlYWYxIiwicyI6Ik1qWXlZemczWXpNdE4yUTFNQzAwTURrd0xXSmlNR1F0TjJOalptRmhaRGsyWTJNdyJ9"

SERVER_CONFIG = {
    'port': 8000,
    'host': '0.0.0.0',
    'history_limit': 10
}

## Install Dependencies

In [20]:
%pip install -q transformers accelerate peft psycopg2-binary fastapi uvicorn pydantic torch sentence-transformers

## Download Cloudflared

In [21]:
%%bash
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
chmod +x cloudflared
sleep 1
./cloudflared --version

cloudflared version 2025.11.1 (built 2025-11-07-16:59 UTC)


## Download Models from Azure Blob

In [22]:
import os
import zipfile
import urllib.request

if not os.path.exists('data_model'):
    print("Downloading models from Azure...")
    
    zip_path = 'model.zip'
    urllib.request.urlretrieve(MODEL_ZIP_URL, zip_path)
    
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('data_model')
    
    os.remove(zip_path)
    print("✓ Models downloaded and extracted successfully")
else:
    print("Models already exist")

Models already exist


## Load Model + LoRA

In [23]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Loading models...")

use_gpu = torch.cuda.is_available()
device_name = "cuda" if use_gpu else "cpu"
dtype = torch.float16 if use_gpu else torch.float32

print(f"Device: {device_name}")
print(f"Dtype: {dtype}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_CONFIG['base_model'])

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_CONFIG['base_model'],
    torch_dtype=dtype,
    device_map="auto"
)
base_model.eval()

print("Loading finetuned model (base + LoRA)...")
finetuned_model = AutoModelForCausalLM.from_pretrained(
    MODEL_CONFIG['base_model'],
    torch_dtype=dtype,
    device_map="auto"
)

lora_path = f"data_model/model/{MODEL_CONFIG['lora_folder']}"
finetuned_model = PeftModel.from_pretrained(finetuned_model, lora_path, local_files_only=True)
finetuned_model.eval()

print(f"Models loaded successfully on {device_name}!")
if use_gpu:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

Loading models...
Device: cuda
Dtype: torch.float16
Loading base model...
Loading finetuned model (base + LoRA)...
Models loaded successfully on cuda!
GPU: NVIDIA A100-SXM4-40GB
VRAM allocated: 12.40 GB


## Database Functions

In [24]:
import psycopg2
from psycopg2.extras import RealDictCursor

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def create_new_session():
    try:
        conn = get_db_connection()  
        cur = conn.cursor()
        cur.execute("INSERT INTO chat_sessions DEFAULT VALUES RETURNING session_id")
        session_id = cur.fetchone()[0]
        conn.commit()
        cur.close()
        conn.close()
        return session_id
    except Exception as e:
        print(f"[ERROR] Failed to create session: {e}", flush=True)
        raise

def update_session_activity(session_id):
    try:
        conn = get_db_connection()
        cur = conn.cursor()
        cur.execute(
            "UPDATE chat_sessions SET last_active_at = CURRENT_TIMESTAMP WHERE session_id = %s",
            (session_id,)
        )
        conn.commit()
        cur.close()
        conn.close()
    except Exception as e:
        print(f"[ERROR] Failed to update session: {e}", flush=True)

def save_message(session_id, role, content):
    try:
        update_session_activity(session_id)
        conn = get_db_connection()
        cur = conn.cursor()
        cur.execute(
            "INSERT INTO chat_messages (session_id, role, content) VALUES (%s, %s, %s)",
            (session_id, role, content)
        )
        conn.commit()
        cur.close()
        conn.close()
    except Exception as e:
        print(f"[ERROR] Failed to save message: {e}", flush=True)
        raise

def get_history(session_id, limit=None):
    if limit is None:
        limit = SERVER_CONFIG['history_limit']
    try:
        update_session_activity(session_id)
        conn = get_db_connection()
        cur = conn.cursor(cursor_factory=RealDictCursor)
        cur.execute(
            "SELECT role, content FROM chat_messages WHERE session_id = %s ORDER BY created_at DESC LIMIT %s",
            (session_id, limit)
        )
        rows = cur.fetchall()
        cur.close()
        conn.close()
        return list(reversed(rows))
    except Exception as e:
        print(f"[ERROR] Failed to get history: {e}", flush=True)
        return []

In [25]:
def generate_reply(
    prompt: str,
    session_id: int,
    model,
    system_prompt: str = None,
    max_new_tokens: int = 2048,
    temperature: float = 0.3,
    top_p: float = 0.95,
    do_sample: bool = True
) -> str:
    if system_prompt is None:
        system_prompt = MODEL_CONFIG['default_system_prompt']

    history = get_history(session_id)

    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history)
    messages.append({"role": "user", "content": prompt})

    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        return_attention_mask=True
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else 1.0,
        top_p=top_p if do_sample else 1.0,
        do_sample=do_sample,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return response.strip()

In [26]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embedding_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
print("Embedding model loaded (768 dimensions)")

def embed_query(query: str):
    embedding = embedding_model.encode(query, convert_to_numpy=True)
    return embedding.tolist()

def retrieve_chunks(query_embedding, top_k=3, course_id=None, content_id=None):
    conn = get_db_connection()
    cur = conn.cursor(cursor_factory=RealDictCursor)

    if content_id:
        cur.execute(
            """
            SELECT chunk_text, chunk_id, course_id, content_id, start_time, end_time,
                   1 - (embedding <=> %s::vector) as similarity
            FROM transcript_chunks
            WHERE content_id = %s
            ORDER BY embedding <=> %s::vector
            LIMIT %s
            """,
            (query_embedding, content_id, query_embedding, top_k)
        )
    elif course_id:
        cur.execute(
            """
            SELECT chunk_text, chunk_id, course_id, content_id, start_time, end_time,
                   1 - (embedding <=> %s::vector) as similarity
            FROM transcript_chunks
            WHERE course_id = %s
            ORDER BY embedding <=> %s::vector
            LIMIT %s
            """,
            (query_embedding, course_id, query_embedding, top_k)
        )
    else:
        cur.execute(
            """
            SELECT chunk_text, chunk_id, course_id, content_id, start_time, end_time,
                   1 - (embedding <=> %s::vector) as similarity
            FROM transcript_chunks
            ORDER BY embedding <=> %s::vector
            LIMIT %s
            """,
            (query_embedding, query_embedding, top_k)
        )

    chunks = cur.fetchall()
    cur.close()
    conn.close()
    return chunks

def generate_with_rag(query, model, chunks, session_id, system_prompt=None, **kwargs):
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        time_info = ""
        if chunk['start_time'] and chunk['end_time']:
            time_info = f" [{chunk['start_time']:.1f}s - {chunk['end_time']:.1f}s]"
        context_parts.append(f"[Đoạn {i}{time_info}]\n{chunk['chunk_text']}")

    context = "\n\n".join(context_parts)

    rag_prompt = f"""Dựa trên các đoạn transcript video sau đây, hãy trả lời câu hỏi:

{context}

Câu hỏi: {query}

Hãy trả lời dựa trên thông tin trong các đoạn transcript trên."""

    response = generate_reply(
        prompt=rag_prompt,
        session_id=session_id,
        model=model,
        system_prompt=system_prompt or "Bạn là trợ lý AI chuyên phân tích và tổng hợp thông tin từ video học tập.",
        **kwargs
    )

    return response

Loading embedding model...
Embedding model loaded (768 dimensions)


## Load Syllabus Model (Qwen2.5-1.5B-Instruct + LoRA)

In [27]:
print("Loading syllabus model...")

syllabus_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

if syllabus_tokenizer.pad_token is None:
    syllabus_tokenizer.pad_token = syllabus_tokenizer.eos_token
    syllabus_tokenizer.pad_token_id = syllabus_tokenizer.eos_token_id

syllabus_base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=dtype,
    device_map="auto"
)

syllabus_lora_path = "data_model/model/edtech-syllabus-lora"
syllabus_model = PeftModel.from_pretrained(syllabus_base_model, syllabus_lora_path, local_files_only=True)
syllabus_model.eval()

print(f"Syllabus model loaded successfully!")
if use_gpu:
    print(f"Total VRAM allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

Loading syllabus model...
Syllabus model loaded successfully!
Total VRAM allocated: 12.80 GB


In [28]:
def generate_syllabus(instruction: str, max_new_tokens: int = 1024) -> str:
    inputs = syllabus_tokenizer(
        instruction,
        return_tensors="pt",
        padding=True,
        return_attention_mask=True
    ).to(syllabus_model.device)

    outputs = syllabus_model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        pad_token_id=syllabus_tokenizer.pad_token_id,
        eos_token_id=syllabus_tokenizer.eos_token_id
    )

    text = syllabus_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text.strip()

## Load Adaptive Model (Qwen2.5-1.5B-Instruct + LoRA)

In [29]:
print("Loading adaptive model...")

adaptive_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

if adaptive_tokenizer.pad_token is None:
    adaptive_tokenizer.pad_token = adaptive_tokenizer.eos_token
    adaptive_tokenizer.pad_token_id = adaptive_tokenizer.eos_token_id

adaptive_base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=dtype,
    device_map="auto"
)

adaptive_lora_path = "data_model/model/adaptive_lora_qwen"
adaptive_model = PeftModel.from_pretrained(adaptive_base_model, adaptive_lora_path, local_files_only=True)
adaptive_model.eval()

print(f"Adaptive model loaded successfully!")
if use_gpu:
    print(f"Total VRAM allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

Loading adaptive model...
Adaptive model loaded successfully!
Total VRAM allocated: 12.81 GB


## Adaptive Learning Functions

In [30]:
import json
import re
from typing import Dict, Any, List

def adaptive_build_messages(data: Dict[str, Any]) -> list:
    context = data.get("current_context", {})
    kg = data.get("knowledge_graph_subgraph", [])
    profile = data.get("user_mastery_profile", [])

    curr_id = str(context.get('target_content_id'))
    curr_theta = float(context.get('current_theta', 0))

    profile_map = {str(item['content_id']): item for item in profile}
    curr_info = profile_map.get(curr_id, {})
    curr_title = curr_info.get('title', f"Lesson {curr_id}")

    curr_desc = "Không có mô tả nội dung."
    for e in kg:
        if str(e.get('target_id')) == curr_id:
            curr_desc = e.get('description_target', '')
            if curr_desc: break
        elif str(e.get('source_id')) == curr_id:
            curr_desc = e.get('description_source', '')
            if curr_desc: break

    prereq_analysis = []
    for e in kg:
        if str(e['target_id']) == curr_id and e['type'] == 'PREREQUISITE':
            src_id = str(e['source_id'])
            src_title = e.get('source', 'Unknown Lesson')
            src_content = e.get('description_source', 'Không có mô tả chi tiết.')
            p_theta = profile_map.get(src_id, {}).get('theta', 0.0)
            status = "HỔNG KIẾN THỨC" if p_theta < 0 else "NẮM VỮNG"
            prereq_analysis.append(
                f"- Bài: {src_title} (ID: {src_id})\n"
                f"  + Nội dung kiến thức: {src_content}\n"
                f"  + Trạng thái học viên: Theta {p_theta} ({status})"
            )

    next_analysis = []
    for e in kg:
        if str(e['source_id']) == curr_id:
            tgt_id = str(e['target_id'])
            tgt_title = e.get('target', 'Unknown Next Lesson')
            tgt_content = e.get('description_target', 'Nội dung bài tiếp theo')
            next_analysis.append(f"- {tgt_title} (ID: {tgt_id}): {tgt_content}")

    system_prompt = """Bạn là một giảng viên, hãy đưa ra chiến lược giúp học viên hiểu sâu và bền vững hơn về bài học
CHIẾN LƯỢC TƯ DUY:
- Đọc nội dung bài học hiện tại và các bài liên quan để hiểu kiến thức cốt lõi đang được sử dụng.
- So sánh nội dung kiến thức nền tảng với mức độ hiểu hiện tại của học viên.
  Nếu học viên chưa nắm được các khái niệm cần thiết cho bài hiện tại, việc học tiếp sẽ kém hiệu quả → ưu tiên ôn tập.
- Nếu kiến thức nền đã đủ nhưng học viên vẫn gặp khó khăn ở bài hiện tại,
  hãy xem đây là vấn đề về độ khó hoặc khả năng áp dụng → cần củng cố thêm trước khi học mới.
- Chỉ đề xuất học tiếp khi học viên thể hiện sự sẵn sàng về mặt hiểu biết,
  dựa trên mối liên hệ nội dung giữa các bài học, không chỉ dựa vào con số.

OUTPUT FORMAT (JSON Only):
{"Action": "REVIEW/REMEDIAL/NEXT", "Description": "Giải thích dựa trên mối liên hệ nội dung bài học..."}"""

    user_prompt = f"""
--- BỐI CẢNH BÀI ĐANG HỌC ---
Tên bài: {curr_title} (ID: {curr_id})
Mô tả nội dung: {curr_desc}
Kết quả hiện tại: Theta {curr_theta}

--- PHÂN TÍCH CÁC BÀI TIỀN ĐỀ (PREREQUISITES) ---
{chr(10).join(prereq_analysis) if prereq_analysis else "Không có bài tiền đề (Đây là bài nhập môn)."}

--- CÁC LỰA CHỌN TIẾP THEO ---
{chr(10).join(next_analysis) if next_analysis else "Không có dữ liệu bài tiếp theo."}

Dựa trên nội dung và kết quả trên, hãy đưa ra quyết định."""

    return [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]

def adaptive_resolve_target_id(action: str, data: Dict[str, Any]) -> str:
    kg = data.get("knowledge_graph_subgraph", [])
    profile = data.get("user_mastery_profile", [])
    curr_id = str(data["current_context"]['target_content_id'])

    p_map = {str(item['content_id']): float(item['theta']) for item in profile}

    if action == "REVIEW":
        candidates = []
        for e in kg:
            if str(e['target_id']) == curr_id and e['type'] == 'PREREQUISITE':
                src_id = str(e['source_id'])
                theta = p_map.get(src_id, 0.0)
                if theta < 0:
                    candidates.append((src_id, theta))

        if candidates:
            candidates.sort(key=lambda x: x[1])
            return candidates[0][0]
        return curr_id

    if action == "NEXT":
        for e in kg:
            if str(e['source_id']) == curr_id and e['type'] == "RELATED":
                return str(e['target_id'])

    return curr_id

def adaptive_fallback_logic(data: Dict[str, Any]) -> Dict[str, Any]:
    ctx = data.get("current_context", {})
    kg = data.get("knowledge_graph_subgraph", [])
    profile = data.get("user_mastery_profile", [])

    curr_id = str(ctx.get('target_content_id'))
    curr_theta = float(ctx.get('current_theta', 0))

    p_map = {str(i['content_id']): float(i['theta']) for i in profile}

    prereq_candidates = []
    for e in kg:
        if str(e.get('target_id')) == curr_id and e.get('type') == 'PREREQUISITE':
            prereq_id = str(e.get('source_id'))
            theta = p_map.get(prereq_id, 0.0)
            if theta < 0:
                prereq_candidates.append((prereq_id, theta, e))

    if prereq_candidates:
        prereq_candidates.sort(key=lambda x: x[1])
        prereq_id, theta, edge = prereq_candidates[0]
        return {
            "Action": "REVIEW",
            "TargetID": prereq_id,
            "Description": f"Phát hiện hổng kiến thức nền tảng '{edge.get('source', '')}'"
        }

    if curr_theta < -1.0:
        for e in kg:
            if str(e.get('source_id')) == curr_id and e.get('type') == 'REMEDIAL':
                return {
                    "Action": "REMEDIAL",
                    "TargetID": str(e.get('target_id')),
                    "Description": "Đề xuất ôn tập bài remedial do năng lực hiện tại thấp"
                }

        return {
            "Action": "REMEDIAL",
            "TargetID": curr_id,
            "Description": "Đề xuất ôn tập lại bài hiện tại do năng lực còn yếu"
        }

    for e in kg:
        if str(e.get('source_id')) == curr_id and e.get('type') == 'RELATED':
            return {
                "Action": "NEXT",
                "TargetID": str(e.get('target_id')),
                "Description": "Đủ điều kiện chuyển sang bài học tiếp theo"
            }

    return {
        "Action": "NEXT",
        "TargetID": curr_id,
        "Description": "Không tìm thấy lựa chọn phù hợp hơn, giữ nguyên bài hiện tại"
    }

def adaptive_predict(request_data: Dict[str, Any]) -> Dict[str, Any]:
    request_id = request_data.get("request_id")

    try:
        messages = adaptive_build_messages(request_data)

        text = adaptive_tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False
        )

        inputs = adaptive_tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            return_attention_mask=True
        ).to(adaptive_model.device)

        outputs = adaptive_model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=256,
            temperature=0.0,
            do_sample=False,
            pad_token_id=adaptive_tokenizer.pad_token_id,
            eos_token_id=adaptive_tokenizer.eos_token_id,
        )

        input_length = inputs.input_ids.shape[1]
        generated_tokens = outputs[0][input_length:]
        response_text = adaptive_tokenizer.decode(generated_tokens, skip_special_tokens=True)

        match = re.search(r'\{[\s\S]*\}', response_text)
        if match:
            ai_result = json.loads(match.group(0))
            action = ai_result.get("Action", "NEXT").upper()
            description = ai_result.get("Description", "AI Recommendation")

            final_target_id = adaptive_resolve_target_id(action, request_data)

            if action == "REVIEW" and final_target_id == str(request_data["current_context"]['target_content_id']):
                action = "REMEDIAL"
                description = "Đề xuất ôn tập bài hiện tại."

            return {
                "request_id": request_id,
                "suggestion": {
                    "Action": action,
                    "TargetID": final_target_id,
                    "Description": description
                },
                "status": "SUCCESS"
            }
        else:
            raise Exception("Invalid JSON Format")

    except Exception as e:
        print(f"Adaptive fallback triggered: {e}")
        fallback = adaptive_fallback_logic(request_data)
        return {
            "request_id": request_id,
            "suggestion": fallback,
            "status": "SUCCESS",
            "debug_info": str(e)
        }

print("Adaptive learning functions ready!")

Adaptive learning functions ready!


## FastAPI Server

In [31]:
import threading
from typing import Optional, List
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import html as html_lib

app = FastAPI()

class ChatRequest(BaseModel):
    prompt: str
    session_id: Optional[int] = None
    system: Optional[str] = None
    max_new_tokens: Optional[int] = 2048
    temperature: Optional[float] = 0.3
    top_p: Optional[float] = 0.95
    do_sample: Optional[bool] = True

class RAGRequest(BaseModel):
    prompt: str
    session_id: Optional[int] = None
    course_id: Optional[int] = None
    content_id: Optional[int] = None
    top_k: Optional[int] = 3
    system: Optional[str] = None
    max_new_tokens: Optional[int] = 2048
    temperature: Optional[float] = 0.3
    top_p: Optional[float] = 0.95
    do_sample: Optional[bool] = True

class SyllabusRequest(BaseModel):
    instruction: str
    max_new_tokens: Optional[int] = 1024

class CurrentContext(BaseModel):
    target_content_id: str
    current_theta: float

class KnowledgeGraphEdge(BaseModel):
    source_id: str
    source: str
    description_source: Optional[str] = ""
    target_id: str
    target: str
    type: str
    description_target: Optional[str] = ""

class UserMastery(BaseModel):
    content_id: str
    title: Optional[str] = None
    theta: float
    status: str

class AdaptiveRequest(BaseModel):
    request_id: str
    current_context: CurrentContext
    knowledge_graph_subgraph: List[KnowledgeGraphEdge]
    user_mastery_profile: List[UserMastery]

def format_response_html(text: str) -> str:
    if '```' in text:
        result = []
        in_code_block = False
        code_lang = 'python'
        code_lines = []

        lines = text.split('\n')
        i = 0
        while i < len(lines):
            line = lines[i]

            if line.startswith('```'):
                if not in_code_block:
                    in_code_block = True
                    lang_match = line[3:].strip()
                    code_lang = lang_match if lang_match else 'python'
                    code_lines = []
                else:
                    in_code_block = False
                    code_content = '\n'.join(code_lines)
                    escaped = html_lib.escape(code_content)
                    result.append(f'<pre><code class="language-{code_lang}">{escaped}</code></pre>')
                    code_lines = []
                i += 1
                continue

            if in_code_block:
                code_lines.append(line)
            else:
                if line.strip():
                    escaped_line = html_lib.escape(line)
                    result.append(f'<p>{escaped_line}</p>')
                else:
                    result.append('<br>')

            i += 1

        if in_code_block:
            code_content = '\n'.join(code_lines)
            escaped = html_lib.escape(code_content)
            result.append(f'<pre><code class="language-{code_lang}">{escaped}</code></pre>')

        return ''.join(result)
    else:
        escaped = html_lib.escape(text)
        escaped = escaped.replace('\n', '<br>')
        return f'<div>{escaped}</div>'

@app.post("/finetune")
def finetune_endpoint(body: ChatRequest):
    try:
        session_id = body.session_id if body.session_id else create_new_session()

        response = generate_reply(
            prompt=body.prompt,
            session_id=session_id,
            model=finetuned_model,
            system_prompt=body.system,
            max_new_tokens=body.max_new_tokens,
            temperature=body.temperature,
            top_p=body.top_p,
            do_sample=body.do_sample
        )

        return {
            "status": 200,
            "message": "Request processed successfully",
            "data": {
                "session_id": session_id,
                "response_raw": response,
                "response_html": format_response_html(response)
            }
        }
    except Exception as e:
        return {
            "status": 500,
            "message": str(e),
            "data": None
        }

@app.post("/base")
def base_endpoint(body: ChatRequest):
    try:
        session_id = body.session_id if body.session_id else create_new_session()

        response = generate_reply(
            prompt=body.prompt,
            session_id=session_id,
            model=base_model,
            system_prompt=body.system,
            max_new_tokens=body.max_new_tokens,
            temperature=body.temperature,
            top_p=body.top_p,
            do_sample=body.do_sample
        )

        return {
            "status": 200,
            "message": "Request processed successfully",
            "data": {
                "session_id": session_id,
                "response_raw": response,
                "response_html": format_response_html(response)
            }
        }
    except Exception as e:
        return {
            "status": 500,
            "message": str(e),
            "data": None
        }

@app.post("/base-rag")
def base_rag_endpoint(body: RAGRequest):
    try:
        session_id = body.session_id if body.session_id else create_new_session()

        query_embedding = embed_query(body.prompt)
        chunks = retrieve_chunks(query_embedding, body.top_k, body.course_id, body.content_id)

        response = generate_with_rag(
            query=body.prompt,
            model=base_model,
            chunks=chunks,
            session_id=session_id,
            system_prompt=body.system,
            max_new_tokens=body.max_new_tokens,
            temperature=body.temperature,
            top_p=body.top_p,
            do_sample=body.do_sample
        )

        return {
            "status": 200,
            "message": "Request processed successfully",
            "data": {
                "session_id": session_id,
                "response_raw": response,
                "response_html": format_response_html(response),
                "chunks": [
                    {
                        "chunk_id": int(c['chunk_id']),
                        "chunk_text": c['chunk_text'],
                        "course_id": int(c['course_id']) if c['course_id'] else None,
                        "content_id": int(c['content_id']),
                        "start_time": float(c['start_time']) if c['start_time'] else None,
                        "end_time": float(c['end_time']) if c['end_time'] else None,
                        "similarity": float(c['similarity'])
                    }
                    for c in chunks
                ]
            }
        }
    except Exception as e:
        return {
            "status": 500,
            "message": str(e),
            "data": None
        }

@app.post("/finetune-rag")
def finetune_rag_endpoint(body: RAGRequest):
    try:
        session_id = body.session_id if body.session_id else create_new_session()

        query_embedding = embed_query(body.prompt)
        chunks = retrieve_chunks(query_embedding, body.top_k, body.course_id, body.content_id)

        response = generate_with_rag(
            query=body.prompt,  
            model=finetuned_model,
            chunks=chunks,
            session_id=session_id,
            system_prompt=body.system,
            max_new_tokens=body.max_new_tokens,
            temperature=body.temperature,
            top_p=body.top_p,
            do_sample=body.do_sample
        )

        return {
            "status": 200,
            "message": "Request processed successfully",
            "data": {
                "session_id": session_id,
                "response_raw": response,
                "response_html": format_response_html(response),
                "chunks": [
                    {
                        "chunk_id": int(c['chunk_id']),
                        "chunk_text": c['chunk_text'],
                        "course_id": int(c['course_id']) if c['course_id'] else None,
                        "content_id": int(c['content_id']),
                        "start_time": float(c['start_time']) if c['start_time'] else None,
                        "end_time": float(c['end_time']) if c['end_time'] else None,
                        "similarity": float(c['similarity'])
                    }
                    for c in chunks
                ]
            }
        }
    except Exception as e:
        return {
            "status": 500,
            "message": str(e),
            "data": None
        }

@app.post("/generate-syllabus")
def generate_syllabus_endpoint(body: SyllabusRequest):
    try:
        instruction = body.instruction.strip()
        if not instruction:
            return {
                "status": 400,
                "message": "Instruction cannot be empty",
                "data": None
            }

        syllabus_raw = generate_syllabus(instruction, body.max_new_tokens)
        syllabus_html = format_response_html(syllabus_raw)

        return {
            "status": 200,
            "message": "Syllabus generated successfully",
            "data": {
                "syllabus_raw": syllabus_raw,
                "syllabus_html": syllabus_html
            }
        }
    except Exception as e:
        return {
            "status": 500,
            "message": str(e),
            "data": None
        }

@app.post("/adaptive")
def adaptive_endpoint(body: AdaptiveRequest):
    try:
        input_data = body.model_dump()
        result = adaptive_predict(input_data)
        return result
    except Exception as e:
        return {
            "request_id": body.request_id,
            "suggestion": None,
            "status": "ERROR",
            "message": str(e)
        }

config = uvicorn.Config(
    app,
    host=SERVER_CONFIG['host'],
    port=SERVER_CONFIG['port'],
    log_level="info"
)
server = uvicorn.Server(config)
server.install_signal_handlers = lambda: None
api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()
print(f"API server đang chạy trên cổng {SERVER_CONFIG['port']}")
print(f"  - POST /finetune (model finetuned)")
print(f"  - POST /base (model gốc)")
print(f"  - POST /base-rag (model gốc + RAG)")
print(f"  - POST /finetune-rag (model finetuned + RAG)")
print(f"  - POST /generate-syllabus (syllabus model)")
print(f"  - POST /adaptive (adaptive learning)")

API server đang chạy trên cổng 8000
  - POST /finetune (model finetuned)
  - POST /base (model gốc)
  - POST /base-rag (model gốc + RAG)
  - POST /finetune-rag (model finetuned + RAG)
  - POST /generate-syllabus (syllabus model)
  - POST /adaptive (adaptive learning)


INFO:     Started server process [1163]
INFO:     Waiting for application startup.


INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


## Start Cloudflared Tunnel

In [32]:
import subprocess

proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "run", "--token", CLOUDFLARED_TOKEN],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

def _stream_output(p):
    for line in p.stdout:
        print(line, end="")

threading.Thread(target=_stream_output, args=(proc,), daemon=True).start()
print("Tunnel đang chạy, giữ kernel hoạt động để kết nối không bị dừng.")

Tunnel đang chạy, giữ kernel hoạt động để kết nối không bị dừng.


2026-01-03T18:29:57Z INF Starting tunnel tunnelID=c8aecbce-c11a-48c0-aaa8-c38b7a6ceaf1
2026-01-03T18:29:57Z INF Version 2025.11.1 (Checksum 991dffd8889ee9f0147b6b48933da9e4407e68ea8c6d984f55fa2d3db4bb431d)
2026-01-03T18:29:57Z INF GOOS: linux, GOVersion: go1.24.9, GoArch: amd64
2026-01-03T18:29:57Z INF Settings: map[token:*****]
2026-01-03T18:29:57Z INF Autoupdate frequency is set autoupdateFreq=86400000
2026-01-03T18:29:57Z INF Generated Connector ID: 6bae7266-d69c-417b-a5cd-839e63b2fff6
2026-01-03T18:29:57Z INF Initial protocol quic
2026-01-03T18:29:57Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-01-03T18:29:57Z INF ICMP proxy will use ::1 in zone lo as source for IPv6


## Stop Server & Tunnel (Uncomment to use)

In [33]:
# Cell mới để debug
print("=== SYSTEM CHECK ===")
print(f"1. Models loaded: {base_model is not None}")
print(f"2. Server port: {SERVER_CONFIG['port']}")
print(f"3. DB config: {DB_CONFIG['host']}:{DB_CONFIG['port']}")

import socket
def check_port(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

print(f"4. Server listening: {check_port(SERVER_CONFIG['port'])}")


2026-01-03T18:29:57Z ERR Cannot determine default origin certificate path. No file cert.pem in [~/.cloudflared ~/.cloudflare-warp ~/cloudflare-warp /etc/cloudflared /usr/local/etc/cloudflared]. You need to specify the origin certificate path by specifying the origincert option in the configuration file, or set TUNNEL_ORIGIN_CERT environment variable originCertPath=
2026-01-03T18:29:57Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-01-03T18:29:57Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-01-03T18:29:57Z INF Starting metrics server on 127.0.0.1:20241/metrics
2026-01-03T18:29:57Z INF Tunnel connection curve preferences: [X25519MLKEM768 CurveP256] connIndex=0 event=0 ip=198.41.192.167
2026/01/03 18:29:57 failed to sufficiently increase receive buffer size (was: 208 kiB, wanted: 7168 kiB, got: 416 kiB). See https://github.com/quic-go/quic-go/wiki/UDP-Buffer-Sizes for details.
=== SYSTEM CHECK ===
1. Models loaded: True
2. Server port: 8000
3. DB config: 2

2026-01-03T18:29:57Z INF Registered tunnel connection connIndex=0 connection=7e004d25-00a3-441d-8ebd-2448b4596e95 event=0 ip=198.41.192.167 location=ord14 protocol=quic
2026-01-03T18:29:57Z INF Tunnel connection curve preferences: [X25519MLKEM768 CurveP256] connIndex=1 event=0 ip=198.41.200.113
2026-01-03T18:29:57Z INF Updated to new configuration config="{\"ingress\":[{\"hostname\":\"ai.nguyenduc.click\",\"id\":\"0\",\"originRequest\":{},\"service\":\"http://127.0.0.1:8000\"},{\"hostname\":\"rag.nguyenduc.click\",\"id\":\"6\",\"originRequest\":{},\"service\":\"http://127.0.0.1:8001\"},{\"hostname\":\"file.nguyenduc.click\",\"id\":\"1\",\"originRequest\":{},\"service\":\"http://localhost:8889\"},{\"hostname\":\"postgres.nguyenduc.click\",\"id\":\"2\",\"originRequest\":{},\"service\":\"http://localhost:5432\"},{\"hostname\":\"ssh.nguyenduc.click\",\"id\":\"3\",\"originRequest\":{},\"service\":\"ssh://localhost:22\"},{\"hostname\":\"nguyenduc.click\",\"id\":\"4\",\"originRequest\":{\"noT

In [ ]:
# try:
#     proc.terminate()
#     proc.wait(timeout=10)
#     print("Đã dừng tunnel")
# except Exception:
#     pass

# try:
#     server.should_exit = True
#     api_thread.join(timeout=5)
#     print("Đã dừng API server")
# except Exception:
#     pass

2026-01-03T18:29:57Z INF Registered tunnel connection connIndex=1 connection=4cc170a4-3bf8-4c21-8029-d60e3b7c1f52 event=0 ip=198.41.200.113 location=ord08 protocol=quic


2026-01-03T18:29:58Z INF Tunnel connection curve preferences: [X25519MLKEM768 CurveP256] connIndex=2 event=0 ip=198.41.192.67
2026-01-03T18:29:58Z INF Registered tunnel connection connIndex=2 connection=f08d4791-9b85-4d61-9927-1480919f1739 event=0 ip=198.41.192.67 location=ord10 protocol=quic
2026-01-03T18:29:59Z INF Tunnel connection curve preferences: [X25519MLKEM768 CurveP256] connIndex=3 event=0 ip=198.41.200.33
2026-01-03T18:30:00Z INF Registered tunnel connection connIndex=3 connection=181db0dd-d00a-4c09-b9fd-4569cae5c087 event=0 ip=198.41.200.33 location=ord08 protocol=quic
INFO:     113.160.14.17:0 - "POST /finetune-rag HTTP/1.1" 422 Unprocessable Entity
INFO:     113.160.14.17:0 - "POST /finetune-rag HTTP/1.1" 422 Unprocessable Entity
INFO:     113.160.14.17:0 - "POST /finetune-rag HTTP/1.1" 200 OK
INFO:     113.160.14.17:0 - "POST /finetune-rag HTTP/1.1" 200 OK
INFO:     113.160.14.17:0 - "POST /finetune-rag HTTP/1.1" 200 OK
INFO:     113.160.14.17:0 - "POST /finetune-rag HTT